In [56]:
import numpy as np
import pandas as pd
from itertools import product

In [92]:
def get_case_3_interval(T_ell, T_h, n_ell, sigma_ratio, eps_ell):
    """
    Compute roots of quadratic
    eps_h ** 2 - (M + n_ell + eps_ell) * eps_h + M * sigma_ratio * (n_ell + eps_ell)
    where M := (T_ell * n_ell) / T_h
    """
    M = (T_ell * n_ell) / T_h
    a = 1
    b = -(M + n_ell + eps_ell)
    c = M * sigma_ratio * (n_ell + eps_ell)

    discr = b ** 2 - 4 * a * c
    if discr < 0:
        return None, None

    eps_h_minus = (-b - np.sqrt(discr)) / (2 * a)
    eps_h_plus = (-b + np.sqrt(discr)) / (2 * a)
    return eps_h_minus, eps_h_plus

def get_eps_l_bar_s(K, F, T_h, sigma_ell, sigma_h, n_ell):
    eps_l_bar_s = K * (F - T_h * K) * sigma_ell / (F * sigma_h - T_h * K * sigma_ell) - n_ell
    return eps_l_bar_s

def get_case_5_threshold(e_l_bar_s, sigma_ell, n_ell, eps_ell, K, n_h, sigma_h, T_l, T_h, F):
    if eps_ell < e_l_bar_s:
        return float("inf")
    e_ell = sigma_ell / (n_ell + eps_ell)
    cand_1 = K - n_h + (n_h * (sigma_h - sigma_ell)) / (K * e_ell - sigma_ell)
    cand_2 = T_l * n_ell * sigma_h / ((F - T_h * K) * e_ell + T_h * sigma_ell)
    return min(cand_1, cand_2)


In [101]:
T_HS = [1, 2, 3]
N_HS = list(np.arange(4, 7.25, 0.25))
N_ELLS = list(np.arange(1, 4, 0.25))
SIGMA_RATIOS = list(np.arange(1.1, 5.1, 0.1))
EPS_ELLS = list(np.arange(0, 1.0, 0.1))

SIGMA_ELL = 50000
K = 9.0
F = 30
T = 14

rows = []
for x in list(product(T_HS, N_HS, N_ELLS, SIGMA_RATIOS, EPS_ELLS)):
    T_h, n_h, n_ell, sigma_ratio, eps_ell = x
    T_ell = T - T_h
    eps_sat = K - n_h

    # check if case 3 -> case 2 switch is possible
    eps_h_minus, eps_h_plus = get_case_3_interval(T_ell, T_h, n_ell, sigma_ratio, eps_ell)
    has_real_roots = eps_h_plus is not None
    upper_root_exceeds_eps_sat = has_real_roots and (eps_h_plus > eps_sat)

    # check if case 3 -> case 4 switch is possible
    sigma_h = sigma_ratio * SIGMA_ELL
    eps_l_bar_s = get_eps_l_bar_s(K, F, T_h, SIGMA_ELL, sigma_h, n_ell)
    eps_h_bar = get_case_5_threshold(eps_l_bar_s, SIGMA_ELL, n_ell, eps_ell, K, n_h, sigma_h, T_ell, T_h, F)
    switch_possible = has_real_roots and (eps_h_plus > eps_sat) and (eps_sat < eps_h_bar)

    rows.append(dict(
        T_h=T_h, T_ell=T_ell, n_h=n_h, n_ell=n_ell,
        sigma_ratio=sigma_ratio, eps_ell=eps_ell,
        eps_sat=eps_sat,
        has_real_roots=has_real_roots,
        case_32_impossible=upper_root_exceeds_eps_sat,
        case_34_possible=switch_possible
    ))

df = pd.DataFrame(rows)

In [102]:
df

,T_h,T_ell,n_h,n_ell,sigma_ratio,eps_ell,eps_sat,has_real_roots,case_32_impossible,case_34_possible
0,1,13,4.0,1.00,1.1,0.0,5.0,True,True,True
1,1,13,4.0,1.00,1.1,0.1,5.0,True,True,True
2,1,13,4.0,1.00,1.1,0.2,5.0,True,True,True
3,1,13,4.0,1.00,1.1,0.3,5.0,True,True,True
4,1,13,4.0,1.00,1.1,0.4,5.0,True,True,True
...,...,...,...,...,...,...,...,...,...,...
187195,3,11,7.0,3.75,5.0,0.5,2.0,False,False,False
187196,3,11,7.0,3.75,5.0,0.6,2.0,False,False,False
187197,3,11,7.0,3.75,5.0,0.7,2.0,False,False,False
187198,3,11,7.0,3.75,5.0,0.8,2.0,False,False,False


In [105]:
print("Total scenarios:", len(df))
print("Scenarios with a real inefficient interval (case 3):", df['has_real_roots'].sum())

sub = df[df['has_real_roots']]
print(
    "Of those, share where upper root exceeds saturation cap:",
    sub['case_32_impossible'].mean()
)

print("\nBy sigma_ratio — share of scenarios with a real inefficient interval:")
print(df.groupby('sigma_ratio')['has_real_roots'].mean())

print("\nAmong those, share where case 3->case 2 is impossible (saturates first):")
print(sub.groupby('sigma_ratio')['case_32_impossible'].mean())

print("\nAmong those, share where case 3->case 4 is possible:")
print(df.groupby('sigma_ratio')['case_34_possible'].mean())

Total scenarios: 187200
Scenarios with a real inefficient interval (case 3): 50362
Of those, share where upper root exceeds saturation cap: 0.9927326158611651

By sigma_ratio — share of scenarios with a real inefficient interval:
sigma_ratio
1.1    1.000000
1.2    0.977778
1.3    0.905556
1.4    0.769444
1.5    0.652778
1.6    0.625000
1.7    0.583333
1.8    0.500000
1.9    0.436111
2.0    0.372222
2.1    0.333333
2.2    0.333333
2.3    0.330556
2.4    0.327778
2.5    0.322222
2.6    0.313889
2.7    0.302778
2.8    0.291667
2.9    0.275000
3.0    0.252778
3.1    0.222222
3.2    0.186111
3.3    0.150000
3.4    0.116667
3.5    0.088889
3.6    0.058333
3.7    0.033333
3.8    0.000000
3.9    0.000000
4.0    0.000000
4.1    0.000000
4.2    0.000000
4.3    0.000000
4.4    0.000000
4.5    0.000000
4.6    0.000000
4.7    0.000000
4.8    0.000000
4.9    0.000000
5.0    0.000000
Name: has_real_roots, dtype: float64

Among those, share where case 3->case 2 is impossible (saturates first):
sigma_r

In [90]:
sub.query('not upper_root_exceeds_eps_sat and T_h == 2')

,T_h,T_ell,n_h,n_ell,sigma_ratio,eps_ell,eps_sat,has_real_roots,upper_root_exceeds_eps_sat
62428,2,12,4.00,1.00,1.3,0.8,5.00,True,False
62429,2,12,4.00,1.00,1.3,0.9,5.00,True,False
62435,2,12,4.00,1.00,1.4,0.5,5.00,True,False
62436,2,12,4.00,1.00,1.4,0.6,5.00,True,False
62437,2,12,4.00,1.00,1.4,0.7,5.00,True,False
62438,2,12,4.00,1.00,1.4,0.8,5.00,True,False
62443,2,12,4.00,1.00,1.5,0.3,5.00,True,False
62444,2,12,4.00,1.00,1.5,0.4,5.00,True,False
62445,2,12,4.00,1.00,1.5,0.5,5.00,True,False
62446,2,12,4.00,1.00,1.5,0.6,5.00,True,False
